# UQ-SHRED Demo: Uncertainty Quantification for SHRED

This notebook demonstrates:
1. **Part 1:** Standard SHRED on SST data (baseline)
2. **Part 2:** UQ-SHRED with uncertainty quantification

We compare reconstruction quality, show uncertainty bands, and evaluate calibration.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.preprocessing import MinMaxScaler

from processdata import load_data, TimeSeriesDataset
from models import SHRED, UQ_SHRED, UQ_Forecaster, fit, fit_uq, forecast_uq
import uq

# Settings
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

## Data Preparation (SST)

In [ ]:
# Load SST data
num_sensors = 3
lags = 52  # 1 year of weekly data

load_X = load_data('SST')
n, m = load_X.shape
print(f'SST data shape: {load_X.shape} (timesteps, spatial dim)')

# Random sensor locations
sensor_locations = np.random.choice(m, size=num_sensors, replace=False)
print(f'Sensor locations: {sensor_locations}')

In [ ]:
# Train/valid/test split
train_indices = np.random.choice(n - lags, size=1000, replace=False)
mask = np.ones(n - lags)
mask[train_indices] = 0
valid_test_indices = np.arange(0, n - lags)[np.where(mask != 0)[0]]
valid_indices = valid_test_indices[::2]
test_indices = valid_test_indices[1::2]

print(f'Train: {len(train_indices)}, Valid: {len(valid_indices)}, Test: {len(test_indices)}')

In [ ]:
# Normalize
sc = MinMaxScaler()
sc = sc.fit(load_X[train_indices])
transformed_X = sc.transform(load_X)

# Generate input sequences
all_data_in = np.zeros((n - lags, lags, num_sensors))
for i in range(len(all_data_in)):
    all_data_in[i] = transformed_X[i:i+lags, sensor_locations]

# Create tensors
train_data_in = torch.tensor(all_data_in[train_indices], dtype=torch.float32).to(device)
valid_data_in = torch.tensor(all_data_in[valid_indices], dtype=torch.float32).to(device)
test_data_in = torch.tensor(all_data_in[test_indices], dtype=torch.float32).to(device)

train_data_out = torch.tensor(transformed_X[train_indices + lags - 1], dtype=torch.float32).to(device)
valid_data_out = torch.tensor(transformed_X[valid_indices + lags - 1], dtype=torch.float32).to(device)
test_data_out = torch.tensor(transformed_X[test_indices + lags - 1], dtype=torch.float32).to(device)

train_dataset = TimeSeriesDataset(train_data_in, train_data_out)
valid_dataset = TimeSeriesDataset(valid_data_in, valid_data_out)
test_dataset = TimeSeriesDataset(test_data_in, test_data_out)

print(f'Input shape: {train_data_in.shape}')
print(f'Output shape: {train_data_out.shape}')

---
# Part 1: Standard SHRED (Baseline)

In [ ]:
# Train SHRED
shred = SHRED(num_sensors, m, hidden_size=64, hidden_layers=2, l1=350, l2=400, dropout=0.1).to(device)
shred_errors = fit(shred, train_dataset, valid_dataset, batch_size=64, num_epochs=200, lr=1e-3, verbose=True, patience=5)

In [ ]:
# SHRED test error
shred.eval()
with torch.no_grad():
    shred_recon = shred(test_dataset.X)
    shred_error = torch.linalg.norm(shred_recon - test_dataset.Y) / torch.linalg.norm(test_dataset.Y)
print(f'SHRED Test Relative Error: {shred_error:.4f}')

In [ ]:
# Plot SHRED reconstruction (single sensor)
shred_recon_np = sc.inverse_transform(shred_recon.cpu().numpy())
test_truth_np = sc.inverse_transform(test_dataset.Y.cpu().numpy())

plt.figure(figsize=(12, 4))
idx = sensor_locations[0]
plt.plot(test_truth_np[:100, idx], 'k-', label='Ground Truth')
plt.plot(shred_recon_np[:100, idx], 'b-', label='SHRED')
plt.xlabel('Time')
plt.ylabel('SST')
plt.title(f'SHRED Reconstruction (Location {idx})')
plt.legend()
plt.tight_layout()
plt.savefig('shred_reconstruction.png', dpi=150)
plt.show()

---
# Part 2: UQ-SHRED

In [ ]:
# Train UQ-SHRED
uq_shred = UQ_SHRED(num_sensors, m, hidden_size=64, hidden_layers=2, l1=350, l2=400, dropout=0.1, noise_dim=50).to(device)
uq_errors = fit_uq(uq_shred, train_dataset, valid_dataset, batch_size=64, num_epochs=200, lr=1e-3, verbose=True, patience=5)

In [ ]:
# UQ-SHRED test error (mean prediction)
uq_shred.eval()
mean_recon, std_recon = uq_shred.reconstruct(test_dataset.X, n_samples=100)
uq_error = torch.linalg.norm(mean_recon - test_dataset.Y) / torch.linalg.norm(test_dataset.Y)
print(f'UQ-SHRED Test Relative Error (mean): {uq_error:.4f}')
print(f'Average uncertainty (std): {std_recon.mean():.4f}')

## Figure 1: Reconstruction with Uncertainty Bands

In [ ]:
# Get samples and quantiles
samples = uq_shred.sample(test_dataset.X, n_samples=200)
quantiles = uq_shred.reconstruct_quantiles(test_dataset.X, quantiles=[0.025, 0.5, 0.975], n_samples=200)

# Inverse transform
samples_np = samples.cpu().numpy()
samples_orig = np.array([sc.inverse_transform(s) for s in samples_np])

mean_orig = samples_orig.mean(axis=0)
lower_orig = np.percentile(samples_orig, 2.5, axis=0)
upper_orig = np.percentile(samples_orig, 97.5, axis=0)

In [ ]:
# Plot reconstruction with uncertainty bands
fig, ax = plt.subplots(figsize=(12, 4))
idx = sensor_locations[0]
t = np.arange(100)

ax.fill_between(t, lower_orig[:100, idx], upper_orig[:100, idx], alpha=0.3, color='blue', label='95% CI')
ax.plot(t, mean_orig[:100, idx], 'b-', linewidth=1.5, label='UQ-SHRED Mean')
ax.plot(t, test_truth_np[:100, idx], 'k--', linewidth=1.5, label='Ground Truth')

ax.set_xlabel('Time')
ax.set_ylabel('SST')
ax.set_title(f'UQ-SHRED Reconstruction with Uncertainty (Location {idx})')
ax.legend()
plt.tight_layout()
plt.savefig('uq_shred_reconstruction.png', dpi=150)
plt.show()

## Figure 2: Multi-Sensor Uncertainty Grid

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
t = np.arange(100)

for i, ax in enumerate(axes):
    idx = sensor_locations[i]
    ax.fill_between(t, lower_orig[:100, idx], upper_orig[:100, idx], alpha=0.3, color='blue')
    ax.plot(t, mean_orig[:100, idx], 'b-', linewidth=1)
    ax.plot(t, test_truth_np[:100, idx], 'k--', linewidth=1)
    ax.set_ylabel(f'Sensor {i+1}')

axes[-1].set_xlabel('Time')
fig.suptitle('UQ-SHRED: Multi-Sensor Reconstruction')
plt.tight_layout()
plt.savefig('uq_shred_multisensor.png', dpi=150)
plt.show()

## Figure 3: SHRED vs UQ-SHRED Comparison

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
idx = sensor_locations[0]
t = np.arange(100)

# SHRED
axes[0].plot(t, test_truth_np[:100, idx], 'k-', label='Ground Truth')
axes[0].plot(t, shred_recon_np[:100, idx], 'b-', label='SHRED')
axes[0].set_ylabel('SST')
axes[0].legend()
axes[0].set_title('SHRED (Deterministic)')

# UQ-SHRED
axes[1].fill_between(t, lower_orig[:100, idx], upper_orig[:100, idx], alpha=0.3, color='blue', label='95% CI')
axes[1].plot(t, test_truth_np[:100, idx], 'k-', label='Ground Truth')
axes[1].plot(t, mean_orig[:100, idx], 'b-', label='UQ-SHRED Mean')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('SST')
axes[1].legend()
axes[1].set_title('UQ-SHRED (With Uncertainty)')

plt.tight_layout()
plt.savefig('shred_vs_uqshred.png', dpi=150)
plt.show()

## Metrics: Calibration

In [ ]:
# Calibration scores
cal_scores = uq.calibration_scores(samples, test_dataset.Y, levels=[0.5, 0.7, 0.9, 0.95, 0.99])
print('Calibration (expected → observed):')
for level, obs in cal_scores.items():
    print(f'  {level*100:.0f}% CI → {obs*100:.1f}% coverage')

## Figure 4: Calibration Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
uq.plot_calibration(samples, test_dataset.Y, ax=ax)
ax.set_title('UQ-SHRED Calibration')
plt.tight_layout()
plt.savefig('calibration_diagram.png', dpi=150)
plt.show()

## Metrics: CRPS and Sharpness

In [ ]:
crps_score = uq.crps(samples, test_dataset.Y)
sharp = uq.sharpness(samples, conf=0.95)

print(f'CRPS: {crps_score:.4f} (lower is better)')
print(f'Sharpness (95% CI width): {sharp:.4f} (lower is sharper)')

## Figure 5: Uncertainty vs Error

In [ ]:
# Does uncertainty predict error?
errors = torch.abs(test_dataset.Y - mean_recon).flatten().cpu().numpy()
stds = std_recon.flatten().cpu().numpy()

# Subsample for plotting
idx = np.random.choice(len(errors), min(5000, len(errors)), replace=False)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(stds[idx], errors[idx], alpha=0.1, s=1)

# Trend line
z = np.polyfit(stds[idx], errors[idx], 1)
p = np.poly1d(z)
x_line = np.linspace(stds.min(), stds.max(), 100)
ax.plot(x_line, p(x_line), 'r-', linewidth=2)

corr = np.corrcoef(stds[idx], errors[idx])[0, 1]
ax.set_xlabel('Predicted Uncertainty (σ)')
ax.set_ylabel('Actual Error |y - ŷ|')
ax.set_title(f'Uncertainty vs Error (correlation={corr:.3f})')

plt.tight_layout()
plt.savefig('uncertainty_vs_error.png', dpi=150)
plt.show()

---
# Part 3: Temporal Forecasting with UQ

In [ ]:
# Prepare forecaster training data
# Input: sensors[t-lags:t], Output: sensors[t]
sensor_data = transformed_X[:, sensor_locations]  # (n, num_sensors)

forecast_in = np.zeros((n - lags, lags, num_sensors))
forecast_out = np.zeros((n - lags, num_sensors))
for i in range(n - lags):
    forecast_in[i] = sensor_data[i:i+lags]
    forecast_out[i] = sensor_data[i+lags]

train_fc_in = torch.tensor(forecast_in[train_indices], dtype=torch.float32).to(device)
train_fc_out = torch.tensor(forecast_out[train_indices], dtype=torch.float32).to(device)
valid_fc_in = torch.tensor(forecast_in[valid_indices], dtype=torch.float32).to(device)
valid_fc_out = torch.tensor(forecast_out[valid_indices], dtype=torch.float32).to(device)

train_fc_dataset = TimeSeriesDataset(train_fc_in, train_fc_out)
valid_fc_dataset = TimeSeriesDataset(valid_fc_in, valid_fc_out)

In [ ]:
# Train UQ-Forecaster
forecaster = UQ_Forecaster(input_size=num_sensors, hidden_size=64, hidden_layers=2, noise_dim=50).to(device)
fc_errors = fit_uq(forecaster, train_fc_dataset, valid_fc_dataset, batch_size=64, num_epochs=200, lr=1e-3, verbose=True, patience=5)

In [ ]:
# Generate forecast trajectories
horizon = 50
initial = test_data_in[0:1]  # Start from first test point

mean_traj, std_traj = forecaster.forecast(initial, horizon=horizon, n_samples=100)
mean_traj = mean_traj.squeeze(0).cpu().numpy()  # (horizon, sensors)
std_traj = std_traj.squeeze(0).cpu().numpy()

# Ground truth future sensors
start_idx = test_indices[0] + lags
gt_future = sensor_data[start_idx:start_idx+horizon]

print(f'Forecast shape: {mean_traj.shape}')

## Figure 6: Temporal Forecast with UQ

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
t = np.arange(horizon)
sensor_idx = 0

ax.fill_between(t, mean_traj[:, sensor_idx] - 2*std_traj[:, sensor_idx],
                   mean_traj[:, sensor_idx] + 2*std_traj[:, sensor_idx],
                alpha=0.3, color='blue', label='95% CI')
ax.plot(t, mean_traj[:, sensor_idx], 'b-', label='Forecast Mean')
ax.plot(t, gt_future[:, sensor_idx], 'k--', label='Ground Truth')

ax.set_xlabel('Forecast Horizon')
ax.set_ylabel('Sensor Value')
ax.set_title('UQ-Forecaster: Temporal Prediction with Uncertainty')
ax.legend()
plt.tight_layout()
plt.savefig('temporal_forecast_uq.png', dpi=150)
plt.show()

## Figure 7: Uncertainty Growth Over Horizon

In [ ]:
# Average uncertainty vs horizon
avg_std = std_traj.mean(axis=1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(horizon), avg_std, 'b-o', markersize=3)
ax.set_xlabel('Forecast Horizon')
ax.set_ylabel('Average Uncertainty (σ)')
ax.set_title('Uncertainty Grows with Forecast Horizon')
plt.tight_layout()
plt.savefig('uncertainty_growth.png', dpi=150)
plt.show()

---
# Summary Table

In [ ]:
print('='*60)
print('RESULTS SUMMARY')
print('='*60)
print(f'{"Metric":<30} {"SHRED":<15} {"UQ-SHRED":<15}')
print('-'*60)
print(f'{"Relative Error":<30} {shred_error:.4f}         {uq_error:.4f}')
print(f'{"CRPS":<30} {"—":<15} {crps_score:.4f}')
print(f'{"Sharpness (95% CI)":<30} {"—":<15} {sharp:.4f}')
print(f'{"Coverage (95% CI)":<30} {"—":<15} {cal_scores[0.95]*100:.1f}%')
print(f'{"UQ-Error Correlation":<30} {"—":<15} {corr:.3f}')
print('='*60)

In [ ]:
# Save all figures list
print('\nSaved figures:')
print('  - shred_reconstruction.png')
print('  - uq_shred_reconstruction.png')
print('  - uq_shred_multisensor.png')
print('  - shred_vs_uqshred.png')
print('  - calibration_diagram.png')
print('  - uncertainty_vs_error.png')
print('  - temporal_forecast_uq.png')
print('  - uncertainty_growth.png')